To test the ApplianceRecommender, we should create a structured notebook that validates:

Filtering: Are recommendations restricted to the correct category and capacity?
Similarity: Do the results look technically similar to the input?
Value Logic: Are we seeing "Good Deals" in the recommendations?

In [1]:
import sys
from pathlib import Path

# Adjust path to import from src
root = Path.cwd()
while root != root.parent:
    if (root / "src").exists(): break
    root = root.parent
sys.path.insert(0, str(root))

import pandas as pd
import numpy as np
from src.recommender import ApplianceRecommender
import warnings
warnings.filterwarnings("ignore")

# Initialize Recommender
# This might take a few seconds as it pre-calculates Fair Prices for the pool
recommender = ApplianceRecommender()

### **Test Case 1: Standard AC Recommendation**
We'll test with a 1.5 Ton 5-Star LG AC.

In [5]:
# Load actual training data
x_df = pd.read_parquet(
    r"D:/Study/data_science/Appliance Intelligence Price Tracker and Recommender/models/dataset/X_train.parquet"
)

y_df = pd.read_parquet(
    r"D:/Study/data_science/Appliance Intelligence Price Tracker and Recommender/models/dataset/y_train.parquet"
)
sample_row_idx = 120 # chossing a different index for variety
complete_test_features_dict = x_df.loc[sample_row_idx].to_dict()

observed_price_for_complete_test = float(np.expm1(y_df.loc[sample_row_idx, 'log_price']))

recs_ac = recommender.get_recommendations(complete_test_features_dict, n=10)
recs_df = pd.DataFrame(recs_ac) 

# Display relevant columns to verify
print(f'Original Price:{observed_price_for_complete_test}')
cols_to_show = ['brand_name', 'star_rating', 'capacity_ac_tons', 'actual_price', 'fair_price', 'value_score']
print("Recommendations for 1.5 Ton LG AC:")
recs_df[cols_to_show]

Original Price:31790.000000000004
Recommendations for 1.5 Ton LG AC:


,brand_name,star_rating,capacity_ac_tons,actual_price,fair_price,value_score
0,carrier,3.0,1.5,22990.0,35787.335938,1.556648
1,voltas,3.0,1.5,22990.0,35584.597656,1.547829
2,lloyd,3.0,1.5,25490.0,35007.210938,1.373370
3,panasonic,3.0,1.5,30490.0,40227.546875,1.319369
4,panasonic,3.0,2.0,42990.0,55435.933594,1.289508
5,blue star,3.0,0.8,23290.0,28606.933594,1.228293
6,daikin,3.0,1.0,26251.0,31958.867188,1.217434
7,blue star,3.0,1.0,28272.0,33897.808594,1.198989
8,ifb,3.0,1.5,31990.0,37623.039062,1.176087
9,haier,3.0,1.0,23990.0,32698.369141,1.363000


In [8]:
test_input_ac = {
    'category': 'Refrigerator',
    'brand_name': 'whirlpool',
    'capacity_value': 230,
    'star_rating': 4,
    'has_inverter': 0,
    'ac_split': 0
}

recs_ac = recommender.get_recommendations(test_input_ac, n=10)
recs_df = pd.DataFrame(recs_ac)

# Display relevant columns to verify
cols_to_show = ['brand_name', 'star_rating', 'capacity_ref_litres', 'actual_price', 'fair_price', 'value_score']
print("Recommendations for 1.5 Ton LG AC:")
recs_df[cols_to_show]

Recommendations for 1.5 Ton LG AC:


KeyError: "['capacity_ref_litres'] not in index"

### **Test Case 2: "Value Hunter" Check**
Let's see if the recommender finds products where value_score > 1.1 (meaning the Fair Price is 10%+ higher than the Market Price).

In [15]:
# Check the top recommendation's value status
top_rec = recs_ac[0]
print(f"Top Recommendation Value Score: {top_rec['value_score']:.2f}")
if top_rec['value_score'] > 1.0:
    print("✅ Success: Found a product priced below its fair value.")
else:
    print("ℹ️ Info: Found a close spec match, but not necessarily a discount.")

Top Recommendation Value Score: 1.56
✅ Success: Found a product priced below its fair value.


### **Test Case 3: Cross-Category (Refrigerator)**

In [15]:
test_input_ref = {
    'category': 'WashingMachine',
    'brand_name': 'godrej',
    'capacity_value':15,
    'capacity_ref_litres': 1,
    'ref_double_door': 1,
    'ref_frost_free': 1
}

recs_ref = pd.DataFrame(recommender.get_recommendations(test_input_ref, n=3))
print("\nRecommendations for 250L Samsung Refrigerator:")
recs_ref[['brand_name', 'capacity_ref_liters', 'actual_price', 'fair_price', 'value_score']]


Recommendations for 250L Samsung Refrigerator:


,brand_name,capacity_ref_liters,actual_price,fair_price,value_score
0,lg,0.0,59994.0,64731.960938,1.078974


In [16]:
recs_ref

,rating,category,n_features,brand_name,star_rating,has_star_rating,has_inverter,has_wifi,has_voice_control,has_app_control,...,value_score,similarity_score,recommendation_score,_comp_capacity,_comp_brand,_comp_star,_comp_inverter,_comp_feature,capacity_mode,explanation
0,4.7,WashingMachine,11,lg,0.0,0,1,1,0,1,...,1.078974,0.673222,0.612594,1.0,0.0,0.5,1.0,0.982217,exact,"{'match_reasons': 'Smart: WiFi, App control', ..."
